<div class="alert alert-block alert-success">

# Task 1 in Assessment 1
### Group 12: 

#### - Akhmad Fauzi Nurulhamzah
#### - [teammate]

Date: 16/10/2025

Environment: Python 3.13.7
</div>

<div class="alert alert-block alert-danger">
    
## Table of Contents

</div>

[1. Introduction](#Intro) <br>
[2. Importing Libraries](#Libs) <br>
[3. Examining Patent Files](#Examine) <br>
[4. Loading and Parsing Files](#Load) <br>
$\;\;\;\;$[4.1. Exploring Tags of Files](#Explore_Tags) <br>
$\;\;\;\;$[4.2. Grouping the tags](#Grouping_Tags) <br>
$\;\;\;\;$[4.3. Defining Regular Expressions](#Reg_Exp) <br>
$\;\;\;\;$[4.4. Reading txt Files](#Read_Txt) <br>
$\;\;\;\;$[4.5. Reading xlsx Files](#Read_Xlxs) <br>
$\;\;\;\;$[4.6. Combine txt and xls data together](#Combine_Data) <br>
[5. Further Process The Extracted Data](#Process_Data) <br>
$\;\;\;\;$[5.1. Remove any duplicates](#Remove_Duplicate) <br>
$\;\;\;\;$[5.2. Replaces empty values to ‘none’ across all variables](#Replace_Empty) <br>
$\;\;\;\;$[5.3. Convert all text to lowercase](#Convert_Lowercase) <br>
$\;\;\;\;$[5.4. Time of the review to 'YYYY-MM-DD HH:MM:SS'](#Convert_Date) <br>
$\;\;\;\;$[5.5. Further process for review_texts](#More_Process) <br>
[6.  File Output](#write) <br>
$\;\;\;\;$[6.1. Writing to JSON File](#Write_Json) <br>
$\;\;\;\;$[6.2. Verification of the Generated JSON File](#Check_Json) <br>
$\;\;\;\;$[6.3. Writing CSV file](#Write_Csv) <br>
$\;\;\;\;$[6.4. Verification of the Generated CSV File](#Check_Csv) <br>
[7. Summary](#summary) <br>
[8. References](#Ref) <br>

-------------------------------------

<div class="alert alert-block alert-warning">

## 1.  Introduction  <a class="anchor" name="Intro"></a>
    
</div>

This assessment focuses on extracting data from semi-structured text files. The dataset consists of multiple text files (**group012_[0-13].txt**) and one Excel file (**group012.xlsx**) containing various information about user reviews. The main objective is to extract and transform the data from all input files including one Excel file and several text files formatted as mis-structured XML into clean, structured formats. The final outputs are required to be in both CSV and JSON file formats.

---

<div class="alert alert-block alert-warning">
    
## 2.  Importing Libraries  <a class="anchor" name="Libs"></a>

</div>

The packages to be used in this assessment are imported in the following. They are used to fulfill the following tasks:
* **re**: for text pattern matching, HTML tag removal, and regular expressions.
* **pandas**: for data manipulation, merging, cleaning, and exporting to Excel/CSV.
* **json**: for exporting hierarchical review data into JSON format.

In [1]:
import re
import pandas as pd

To simplify configuration, the code block below is used to define the input and output file paths. This approach ensures flexibility when running the notebook. For example, if a file is missing, renamed, or if the examiner executes the notebook from a different directory. This task is implemented using Jupyter Notebook as the main development environment

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

txt_filename = [f'student_group012/group012_0.txt', f'student_group012/group012_1.txt', f'student_group012/group012_2.txt',
                f'student_group012/group012_3.txt', f'student_group012/group012_4.txt', f'student_group012/group012_5.txt',
                f'student_group012/group012_6.txt', f'student_group012/group012_7.txt', f'student_group012/group012_8.txt',
                f'student_group012/group012_9.txt', f'student_group012/group012_10.txt', f'student_group012/group012_11.txt',
                f'student_group012/group012_12.txt', f'student_group012/group012_13.txt']

xls_filename = f'student_group012/group012.xlsx'

output_json_filename = "task1_012.json"

output_csv_filename = "task1_012.csv"

-------------------------------------

<div class="alert alert-block alert-warning">

## 3.  Examining Raw Data <a class="anchor" name="Examine"></a>

 </div>

Before we begin the data wrangling process, let's first analyze the input files using a text editor. This will help us understand their format and develop a strategy for reading the data for subsequent tasks. The following observations were made:

### **1. General Observations**
* The dataset consists of 15 files in total: 14 text files (.txt) and one Microsoft Excel file (.xlsx).

### **2. Analysis of All 14 Text Files**
* All 14 .txt files are confirmed to be structured as XML (Extensible Markup Language) documents. Each contains a **dataset** root element holding multiple **record** entries. This confirms that using a robust XML parser is the correct strategy.
* Pervasive Inconsistency in Tag Naming & Capitalization: The inconsistency is not isolated to a few files but is present throughout the entire set. A complete scan of all 14 files revealed the following variations for each required data attribute:
  * **Category**: _CaTEGORY_, _CATEGORY_, _category_, _Category_
  * **Reviewer ID**: _reviewerID_, _reviewer_id_, _ReviewerID_, _Reviewer_id_
  * **Rating**: _rating_, _rate_, _Rate_, _Rating_
  * **Review Title**: _review_title_, _Review_title_, _Heading_, _heading_
  * **Review Text**: _text_, _review_text_, _Review_text_
  * **Product ID**: _productID_, _product_id_, _PRODUCTID_, _Product_id_
  * **Parent Product ID**: _Parent_product_id_, _ParentPRoductID_, _parentPRODUCTID_, _parent_product_id_
  * **Timestamp**: _Review_timestamp_, _review_timestamp_, _timestamp_, _Timestamp_, _time_, _Time_, _date_, _Date_
  * **Verified Purchase**: _verified_purchase_, _Verified_purchase_, _Is_verified_purchase_, _is_verified_purchase_
  * **Helpful Votes**: _Likes_, _likes_, _helpful_vote_, _Helpful_votes_, _helpful_votes_, _Votes_, _votes_
* Widespread Malformed XML Syntax: The formatting errors are a consistent problem across all files. The parser must be designed to handle these systematically.
  * Examples of extra spaces: < /ReviewerID>, < /rating>, < /review_title>
  * Examples of incorrect closing slashes: <//CaTEGORY>, <//reviewerID>, <//parentPRODUCTID>

### **3. Analysis of the Excel File (group012.xlsx)**
* **File Format**: The Excel file remains the cleanest data source, containing a single sheet with structured, tabular data.
* **Column Structure**: The column names are consistent within the file: category, reviewer_id, rating, review_title, review_text, attached_images, product_id, parent_product_id, review_timestamp, is_verified_purchase, and helpful_votes.
* **Extraneous Columns**: The spreadsheet contains several empty or unnamed columns (e.g., X4, X5, X6, etc.) that must be removed.
* **Column Name Standardization**: To ensure a seamless merge, the column names in the Excel file must be standardized to match the target schema derived from the XML files (e.g., renaming rating to rate, helpful_votes to likes).


-------------------------------------

<div class="alert alert-block alert-warning"> 

## 4.  Loading and Parsing Files <a class="anchor" name="Load"></a>

</div>

In this section, the files are parsed and processed. First, suitable regular expressions are defined to extract the required information from the input files during the reading process.

-------------------------------------

<div class="alert alert-block alert-info">
    
### 4.1. Exploring Tags of Files<a class="anchor" name="Explore_Tags"></a>

</div>

Before we start defining regular expressions, we need to identify the tag patterns in each text file so that we can determine the correct grouping and craft the appropriate regex expressions for parsing the data.

In [3]:
unique_tags = set()

# loop for each file
for i in range(14):
    file_name = txt_filename[i]
    
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            content = f.read()
            
            # Regex Explained:
            # <[/]? : Matches the opening '<' and an optional closing slash '/'
            # \s* : Matches any whitespace characters (like spaces after the '<')
            # ([a-zA-Z0-9_]+) : This is the capturing group. It finds and captures
            #                   one or more letters, numbers, or underscores, which
            #                   is the tag name itself.
            # re.IGNORECASE : Makes the search case-insensitive.
            found_tags = re.findall(r'<[/]?\s*([a-zA-Z0-9_]+)', content, re.IGNORECASE)
            
            # To normalize the tags, convert them all to lowercase before adding to the set
            normalized_tags = {tag.lower() for tag in found_tags}
            unique_tags.update(normalized_tags)

    except FileNotFoundError:
        print(f"ERROR: Could not find file '{file_name}'. Skipping.")
    except Exception as e:
        print(f"An error occurred while processing {file_name}: {e}")

# Display Results
if unique_tags:
    # Convert the set to a sorted list for clean presentation
    sorted_tags = sorted(list(unique_tags))
    
    # Use pandas for a nice, clean table display
    tags_df = pd.DataFrame(sorted_tags, columns=['Discovered_Tag_Names'])
    
    print("         All Discovered Tags             ")
    print("-----------------------------------------")
    print(tags_df.to_string())

else:
    print("\nNo tags were found. Please check the file paths and content.")

         All Discovered Tags             
-----------------------------------------
    Discovered_Tag_Names
0        attached_images
1                     br
2               category
3                dataset
4                   date
5                heading
6           helpful_vote
7          helpful_votes
8   is_verified_purchase
9                  likes
10     parent_product_id
11       parentproductid
12                  pics
13              pictures
14            product_id
15             productid
16                  rate
17                rating
18                record
19           review_text
20      review_timestamp
21          review_title
22           reviewer_id
23            reviewerid
24                  text
25                  time
26             timestamp
27     verified_purchase
28                 votes


---

<div class="alert alert-block alert-info">
    
### 4.2. Grouping the tags <a class="anchor" name="Grouping_Tags"></a>

</div>

After identifying all the tags from the 14 text files, we can group them to formulate the appropriate regular expressions for the code. Defining the correct regular expressions is crucial to efficiently extract the desired information from the text and ensure accurate data parsing.

| No | Grouping| Tag |
|----------|-----|--------------|
| 1 | parent_product_id | parent_product_id |
| 2 | parent_product_id | parentproductid |
| 3 | category | category |
| 4 | reviewer_id | reviewer_id |
| 5 | reviewer_id | reviewerid |
| 6 | rating | rate |
| 7 | rating | rating |
| 8 | review_title | heading |
| 9 | review_title | review_title |
| 10 | review_text | review_text |
| 11 | review_text | text |
| 12 | attached_images | attached_images |
| 13 | attached_images | pics |
| 14 | attached_images | pictures |
| 15 | product_id | product_id |
| 16 | product_id | productid |
| 17 | review_timestamp | date |
| 18 | review_timestamp | review_timestamp |
| 19 | review_timestamp | time |
| 20 | review_timestamp | timestamp |
| 21 | is_verified_purchase | is_verified_purchase |
| 22 | is_verified_purchase | verified_purchase |
| 23 | helpful_votes | helpful_vote |
| 24 | helpful_votes | helpful_votes |
| 25 | helpful_votes | likes |
| 26 | helpful_votes | votes |
| 27 | not_valid | br |
| 28 | not_valid | dataset |
| 29 | not_valid | record |

<div class="alert alert-block alert-info">
    
### 4.3. Defining Regular Expressions <a class="anchor" name="Reg_Exp"></a>

</div>

After knowing all the possible of tag of the 14 files, now we can define the reg ex pattern for each tag.

In [4]:
# Regex Explained:
# <\s*              : Matches the opening angle bracket < followed by any optional whitespace.
# (?P<tag>category) : Defines a named group called tag that specifically matches the word.
#                     This makes it easier to reference the same tag later in the expression.
# \b[^>]*>          : Matches any additional characters after the tag name until the closing > of the opening tag. 
#                     The \b ensures a word boundary, so only “tag” is matched as the tag name.
# \s*               : Matches any optional whitespace after the opening tag.
# (?P<content>.*?)  : Captures the content between the opening and closing tags as a named group content. 
#                     The .*? is a non-greedy match, ensuring it stops at the nearest closing tag.
# \s*<\s*/+\s*      : Matches the start of the closing tag </ (or even <// if double slashes appear) with optional spaces.
# (?P=tag)          : Refers back to the same tag name captured earlier (in this case, category), ensuring that the opening and closing tags match.
# \s*>              : Matches the closing angle bracket > of the closing tag, allowing optional spaces before it.

pattern_category = r'<\s*(?P<tag>category)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_reviewer_id = r'<\s*(?P<tag>reviewer_id|reviewerid)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_rating = r'<\s*(?P<tag>rate|rating)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_review_title = r'<\s*(?P<tag>heading|review_title)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_review_text = r'<\s*(?P<tag>review_text|text)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_attached_images = r'<\s*(?P<tag>attached_images|pics|pictures)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_product_id = r'<\s*(?P<tag>product_id|productid)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_parent_product_id = r'<\s*(?P<tag>parent_product_id|parentproductid)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_review_timestamp = r'<\s*(?P<tag>date|review_timestamp|time|timestamp)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_is_verified_purchase = r'<\s*(?P<tag>is_verified_purchase|verified_purchase)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'
pattern_helpful_votes = r'<\s*(?P<tag>helpful_vote|helpful_votes|likes|votes)\b[^>]*>\s*(?P<content>.*?)\s*<\s*/+\s*(?P=tag)\s*>'

-------------------------------------

<div class="alert alert-block alert-info">
    
### 4.4. Reading txt Files <a class="anchor" name="Read_Txt"></a>

</div>

Next, each file is parsed using the regular expression patterns defined in the previous block. These patterns are applied to extract data for each column from the text files. The extracted data is then combined and converted into a DataFrame, which serves as the main structure for further data manipulation and analysis.

In [5]:
all_records_data = []

# loop for each file
for i in range(14): 
    file_name = txt_filename[i]
    
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            content = f.read()
            
        # Split the file content by the <record> tag to process each record individually
        records = re.split(r'<record>', content, flags=re.IGNORECASE)

        flags = re.IGNORECASE | re.DOTALL
        
        # The first split item is the XML header, so we skip it
        for record_block in records[1:]:

            record_data = {
                'category': re.compile(pattern_category, flags).search(record_block).group("content").strip(),
                'reviewer_id':  re.compile(pattern_reviewer_id, flags).search(record_block).group("content").strip(),
                'rating':  re.compile(pattern_rating, flags).search(record_block).group("content").strip(),
                'review_title':  re.compile(pattern_review_title, flags).search(record_block).group("content").strip(),
                'review_text':  re.compile(pattern_review_text, flags).search(record_block).group("content").strip(),
                'attached_images':  re.compile(pattern_attached_images, flags).search(record_block).group("content").strip(),
                'product_id': re.compile(pattern_product_id, flags).search(record_block).group("content").strip(),
                'parent_product_id':  re.compile(pattern_parent_product_id, flags).search(record_block).group("content").strip(),
                'review_timestamp':  re.compile(pattern_review_timestamp, flags).search(record_block).group("content").strip(),
                'is_verified_purchase':  re.compile(pattern_is_verified_purchase, flags).search(record_block).group("content").strip(),
                'helpful_votes':  re.compile(pattern_helpful_votes, flags).search(record_block).group("content").strip()
            }
            
            all_records_data.append(record_data)

    except FileNotFoundError:
        print(f"ERROR: Could not find file '{file_name}'. Please ensure it's in the same folder as your notebook.")
    except Exception as e:
        print(f"An unknown error occurred while processing {file_name}: {e}")

df_txt = pd.DataFrame(all_records_data)

print(f"Successfully parsed {len(df_txt)} records from the {len(txt_filename)} text files.")

# Now lets show the top 10 of the data

columns_to_show = ['category', 'reviewer_id', 'rating', 'attached_images', 'product_id', 
                   'parent_product_id', 'review_timestamp', 'is_verified_purchase', 'helpful_votes']

print(df_txt[columns_to_show].head(10).to_string(index=False))

Successfully parsed 262038 records from the 14 text files.
  category                  reviewer_id rating attached_images product_id parent_product_id review_timestamp is_verified_purchase helpful_votes
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0                 B08CMG2PTW        B08CMHLCFY    1641071362151                 True             0
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0                 B096F1JXPQ        B096F13NC1    1632350246898                 True             1
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0                 B08ZM39X9V        B09YTY492C    1627776072950                 True             1
Automotive AELCKP3F7OX2PC4KPC5QMSHFNQYA    4.0                 B00L2G47PK        B00L2G47PK    1442935618000                 True             3
Automotive AEBWZHIHOBSGZBVULL27ZPKRGZDQ    1.0                 B08ZYQ3CGX        B08ZYR9R2M    1648277714402                 True             0
Automotive AF35ZISXEJTTKHQU462EBJTXCCPA    3.0                 B09CYVRCRR    

---

<div class="alert alert-block alert-info">
    
### 4.5. Reading xlsx Files <a class="anchor" name="Read_Xlxs"></a>

</div>

Now, we can proceed to the next step reading the .xlsx file. Based on the previous observations, the Excel file does not require extensive adjustments. After loading the file, we only need to extract the data from each sheet and remove the columns that are not needed. From manually inspecting the file, we can see that all sheets are well organized and share the same header structure, making the data extraction process more straightforward.

In [6]:
# Load the Excel file
excel_data = pd.ExcelFile(xls_filename)

# Define an empty DataFrame to collect all sheets
df_xls = pd.DataFrame()

# Loop through all 14 sheets
for i in range(14):

    # get the name of the sheet
    sheet_name = excel_data.sheet_names[i]

    # Read each sheet
    xls_sheet = pd.read_excel(excel_data, sheet_name=sheet_name)

    # Drop unwanted columns (ignore if not found)
    xls_sheet = xls_sheet.drop(columns=['X4', 'X5', 'X6', 'X7', 'X8'], errors='ignore')

    # Append (concatenate) this sheet to the combined DataFrame
    df_xls = pd.concat([df_xls, xls_sheet], ignore_index=True)

# Show top 10 data
print(df_xls[columns_to_show].head(10).to_string(index=False))

  category                  reviewer_id  rating attached_images product_id parent_product_id  review_timestamp  is_verified_purchase  helpful_votes
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ     5.0             NaN B08CMG2PTW        B08CMHLCFY     1641071362151                  True              0
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ     5.0             NaN B096F1JXPQ        B096F13NC1     1632350246898                  True              1
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ     5.0             NaN B08ZM39X9V        B09YTY492C     1627776072950                  True              1
Automotive AELCKP3F7OX2PC4KPC5QMSHFNQYA     4.0             NaN B00L2G47PK        B00L2G47PK     1442935618000                  True              3
Automotive AEBWZHIHOBSGZBVULL27ZPKRGZDQ     1.0             NaN B08ZYQ3CGX        B08ZYR9R2M     1648277714402                  True              0
Automotive AF35ZISXEJTTKHQU462EBJTXCCPA     3.0             NaN B09CYVRCRR        B09NBM4CGM     1640830022066  

---

<div class="alert alert-block alert-info">
    
### 4.6. Combine txt and xls data together<a class="anchor" name="Combine_Data"></a>

</div>

Now, let’s merge both DataFrames the one obtained from the text files and the one from the Excel file into a single unified DataFrame. This combined dataset will then be used for further processing and analysis.

In [7]:
# combile df_xls and df_txt
df_merged = pd.concat([df_xls, df_txt], ignore_index=True)

# Show top 10 data
print(df_merged[columns_to_show].head(10).to_string(index=False))

  category                  reviewer_id rating attached_images product_id parent_product_id review_timestamp is_verified_purchase helpful_votes
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0             NaN B08CMG2PTW        B08CMHLCFY    1641071362151                 True             0
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0             NaN B096F1JXPQ        B096F13NC1    1632350246898                 True             1
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0             NaN B08ZM39X9V        B09YTY492C    1627776072950                 True             1
Automotive AELCKP3F7OX2PC4KPC5QMSHFNQYA    4.0             NaN B00L2G47PK        B00L2G47PK    1442935618000                 True             3
Automotive AEBWZHIHOBSGZBVULL27ZPKRGZDQ    1.0             NaN B08ZYQ3CGX        B08ZYR9R2M    1648277714402                 True             0
Automotive AF35ZISXEJTTKHQU462EBJTXCCPA    3.0             NaN B09CYVRCRR        B09NBM4CGM    1640830022066                 True       

-------------------------------------

<div class="alert alert-block alert-warning"> 

## 5.  Further Process The Extracted Data <a class="anchor" name="Process_Data"></a>

</div>

In this section, we will perform further data cleaning and manipulation on the dataset obtained from both the text files and the Excel file.

---

<div class="alert alert-block alert-info">
    
### 5.1. Remove any duplicates <a class="anchor" name="Remove_Duplicate"></a>

</div>

In [8]:
print(f"Total Data Before Remove Duplicates: {len(df_merged)} ")

# remove duplicate datta
df_merged = df_merged.drop_duplicates()

print(f"Total Data Before After Duplicates: {len(df_merged)} ")

Total Data Before Remove Duplicates: 266238 
Total Data Before After Duplicates: 265208 


---

<div class="alert alert-block alert-info">
    
### 5.2. Replaces empty values to ‘none’ across all variables <a class="anchor" name="Replace_Empty"></a>

</div>

In [9]:
# Replaces empty values to ‘none’ across all variables
df_merged = df_merged.fillna('none')

# handles empty or whitespace-only strings
df_merged = df_merged.replace(r'^\s*$', 'none', regex=True)  

# Show top 5 data
print(df_merged[columns_to_show].head(5).to_string(index=False))

# Show bot 5 data
print(df_merged[columns_to_show].tail(5).to_string(index=False))

  category                  reviewer_id rating attached_images product_id parent_product_id review_timestamp is_verified_purchase helpful_votes
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0            none B08CMG2PTW        B08CMHLCFY    1641071362151                 True             0
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0            none B096F1JXPQ        B096F13NC1    1632350246898                 True             1
Automotive AHST6VU63LKDJK3RURVE4KBN7DMQ    5.0            none B08ZM39X9V        B09YTY492C    1627776072950                 True             1
Automotive AELCKP3F7OX2PC4KPC5QMSHFNQYA    4.0            none B00L2G47PK        B00L2G47PK    1442935618000                 True             3
Automotive AEBWZHIHOBSGZBVULL27ZPKRGZDQ    1.0            none B08ZYQ3CGX        B08ZYR9R2M    1648277714402                 True             0
  category                  reviewer_id rating attached_images product_id parent_product_id review_timestamp is_verified_purchase helpfu

---

<div class="alert alert-block alert-info">
    
### 5.3. Convert all text to lowercase <a class="anchor" name="Convert_Lowercase"></a>

</div>

In this task, we need to carefully consider the impact of converting all text to lowercase. For most variables, this operation does not cause significant issues. However, for the attached_images field, the image links are case-sensitive, and converting them to lowercase would make the links unusable. Nevertheless, after analyzing the sample output data provided in this assignment, it appears that the attached_images links are already non-functional. Therefore, in this task, all text will be converted to lowercase without any exceptions.

In [10]:
# Convert all text to lowercase
df_merged = df_merged.map(lambda x: x.lower() if isinstance(x, str) else x)

# Show top 10 data
print(df_merged[columns_to_show].head(10).to_string(index=False))

  category                  reviewer_id rating attached_images product_id parent_product_id review_timestamp is_verified_purchase helpful_votes
automotive ahst6vu63lkdjk3rurve4kbn7dmq    5.0            none b08cmg2ptw        b08cmhlcfy    1641071362151                 True             0
automotive ahst6vu63lkdjk3rurve4kbn7dmq    5.0            none b096f1jxpq        b096f13nc1    1632350246898                 True             1
automotive ahst6vu63lkdjk3rurve4kbn7dmq    5.0            none b08zm39x9v        b09yty492c    1627776072950                 True             1
automotive aelckp3f7ox2pc4kpc5qmshfnqya    4.0            none b00l2g47pk        b00l2g47pk    1442935618000                 True             3
automotive aebwzhihobsgzbvull27zpkrgzdq    1.0            none b08zyq3cgx        b08zyr9r2m    1648277714402                 True             0
automotive af35zisxejttkhqu462ebjtxccpa    3.0            none b09cyvrcrr        b09nbm4cgm    1640830022066                 True       

---

<div class="alert alert-block alert-info">

### 5.4. Time of the review to 'YYYY-MM-DD HH:MM:SS' <a class="anchor" name="Convert_Date"></a>

</div>

In [11]:
# Convert to numeric first (handles strings like "1627776072950" safely)
df_merged['review_timestamp'] = pd.to_numeric(df_merged['review_timestamp'], errors='coerce')

# Convert from Unix milliseconds → datetime
df_merged['review_timestamp'] = pd.to_datetime(df_merged['review_timestamp'], unit='ms', utc=True)

# Format to string: 'YYYY-MM-DD HH:MM:SS'
df_merged['review_timestamp'] = df_merged['review_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Show top 10 data
print(df_merged[columns_to_show].head(10).to_string(index=False))

  category                  reviewer_id rating attached_images product_id parent_product_id    review_timestamp is_verified_purchase helpful_votes
automotive ahst6vu63lkdjk3rurve4kbn7dmq    5.0            none b08cmg2ptw        b08cmhlcfy 2022-01-01 21:09:22                 True             0
automotive ahst6vu63lkdjk3rurve4kbn7dmq    5.0            none b096f1jxpq        b096f13nc1 2021-09-22 22:37:26                 True             1
automotive ahst6vu63lkdjk3rurve4kbn7dmq    5.0            none b08zm39x9v        b09yty492c 2021-08-01 00:01:12                 True             1
automotive aelckp3f7ox2pc4kpc5qmshfnqya    4.0            none b00l2g47pk        b00l2g47pk 2015-09-22 15:26:58                 True             3
automotive aebwzhihobsgzbvull27zpkrgzdq    1.0            none b08zyq3cgx        b08zyr9r2m 2022-03-26 06:55:14                 True             0
automotive af35zisxejttkhqu462ebjtxccpa    3.0            none b09cyvrcrr        b09nbm4cgm 2021-12-30 02:07:02       

---

<div class="alert alert-block alert-info">
    
### 5.5. Further process for review_texts <a class="anchor" name="More_Process"></a>

</div>

All text data must be transformed into lowercase, free of HTML tags, emojis, and non-UTF-8 characters, and must be written entirely in English.

In [12]:
def clean_review_text(text):
    if not isinstance(text, str):
        return 'none'

    # Lowercase
    text = text.lower()

    # Remove HTML tags (keep inner content)
    text = re.sub(r'<[^>]+>', '', text)

    # Remove emojis and non-UTF-8 / non-printable symbols
    # Keep basic punctuation, letters, numbers, and spaces
    # force valid utf-8
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    # remove non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # remove □, replacement chars
    text = re.sub(r'[\uFFFD\u25A1\u0000]+', '', text)
    # remove emoji range
    text = re.sub(r'[\u2600-\u26FF\u2700-\u27BF]+', '', text)
    # keep readable symbols
    text = re.sub(r'[^a-z0-9\s.,!?\'"-]', '', text)

    # Strip extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Check proportion of English letters
    letters = re.findall(r'[a-z]', text)
    english_ratio = len(letters) / max(len(text), 1)

    if english_ratio < 0.01:  # threshold = 1%
        return 'none'

    return text

# Apply to your dataframe
df_merged['review_text'] = df_merged['review_text'].apply(clean_review_text)

# Verify the result
print(df_merged[['review_text']].head(5).to_string(index=False))

                                                                                                                                                                                                                                                                                                                                                                                                                                                   review_text
                                                                                                                                                                                                                                                                                                                                                                       this item came on time as promised. placed in trunk and hope we never have to use them.
                                                                                                          

-------------------------------------

<div class="alert alert-block alert-warning"> 

## 6.  File Output <a class="anchor" name="write"></a>

</div>

In this section, we will generate the output files in JSON and CSV formats, following the specific structure and formatting requirements outlined in the assignment.

---

<div class="alert alert-block alert-info">
    
### 6.1. Writing to JSON File <a class="anchor" name="Write_Json"></a>

</div>

In [13]:
# Define which columns belong inside each review
review_fields = [
    'category', 'reviewer_id', 'rating', 'review_title', 'review_text',
    'attached_images', 'product_id', 'review_timestamp',
    'is_verified_purchase', 'helpful_votes'
]

# Build JSON structure grouped by parent_product_id
output = {}

for parent_id, group in df_merged.groupby('parent_product_id', dropna=False):
    reviews = []
    for _, row in group.iterrows():
        review = {col: str(row.get(col, 'none')) for col in review_fields}
        reviews.append(review)
    output[str(parent_id)] = {"reviews": reviews}

# Export to JSON file
with open(output_json_filename, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=4)

print(f"JSON file exported successfully: {output_json_filename}")

JSON file exported successfully: task1_012.json


-------------------------------------

<div class="alert alert-block alert-info">
    
### 6.2. Verification of the Generated JSON File <a class="anchor" name="Check_Json"></a>

</div>

In [14]:
try:
    with open(output_json_filename, "r", encoding="utf-8") as f:
        f.read()
    print("The file is valid UTF-8.")
except UnicodeDecodeError:
    print("The file is NOT UTF-8 encoded.")

The file is valid UTF-8.


---

<div class="alert alert-block alert-info">

### 6.3. Writing CSV file <a class="anchor" name="Write_Csv"></a>

</div>

In [15]:
# Ensure 'review_text' and 'parent_product_id' columns exist
# if 'parent_product_id' not in df_merged.columns or 'review_text' not in df_merged.columns:
# raise ValueError("Missing required columns: 'parent_product_id' or 'review_text' in df_merged")

# Count total reviews and review_text counts per parent_product_id
summary = (
    df_merged.groupby('parent_product_id', dropna=False)
    .agg(
        review_count=('review_text', 'size'),
        review_text_count=('review_text', lambda x: (x.str.strip().str.lower() != 'none').sum())
    )
    .reset_index()
)

# Convert all columns to string (as required)
summary = summary.astype(str)

# Save to CSV
summary.to_csv(output_csv_filename, index=False, encoding='utf-8')

print(f"CSV file created successfully: {output_csv_filename}")

# Show top 10 data
print(summary.head(10).to_string(index=False))

CSV file created successfully: task1_012.csv
parent_product_id review_count review_text_count
       0440224764            1                 1
       0486475468            3                 3
       0892877073            1                 1
       0892878053            1                 1
       0998137340            1                 1
       1563923378            1                 1
       1563923807            2                 2
       1563923939            1                 1
       1563923963            1                 1
       1599691701            1                 1


---

<div class="alert alert-block alert-info">
    
### 6.4. Verification of the Generated CSV File <a class="anchor" name="Check_Csv"></a>

</div>

In [16]:
try:
    with open(output_csv_filename, "r", encoding="utf-8") as f:
        f.read()
    print("The file is valid UTF-8.")
except UnicodeDecodeError:
    print("The file is NOT UTF-8 encoded.")

The file is valid UTF-8.


---

<div class="alert alert-block alert-warning"> 

## 7. Summary <a class="anchor" name="summary"></a>

</div>

In this task, we successfully processed and combined data from 14 text files and 1 Excel file. All files were read, parsed, and cleaned, with data from both sources merged into a single unified DataFrame for further analysis. Throughout the process, several key steps were completed:

* Extracted information from semi-structured text files using regular expressions to handle inconsistent XML-like formatting.
* Read and cleaned the Excel file, ensuring consistent column structures across all sheets.
* Combined the text and Excel data into a single DataFrame for unified processing.
* Applied data cleaning operations, including case normalization, HTML tag removal, and handling of invalid characters.
* Converted all text to lowercase and ensured UTF-8 encoding for consistency.
* Exported the final cleaned dataset into both CSV and JSON formats following the required structure.
* Verified that all exported files were properly UTF-8 encoded and readable.

This consolidated dataset is now ready for further data analysis, visualization, or modeling tasks in subsequent stages of the project.

-------------------------------------

<div class="alert alert-block alert-warning"> 

## 8. References <a class="anchor" name="Ref"></a>

</div>

[1]<a class="anchor" name="ref-2"></a> Exporting Pandas Data Frame to CSV, https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html, Accessed 15/10/2025.

[2]<a class="anchor" name="ref-2"></a> How do I write JSON data to a file, https://stackoverflow.com/questions/12309269/how-do-i-write-json-data-to-a-file, Accessed 16/10/2025.

---
